In [19]:
import json
import sys

import pandas as pd

sys.path.append("../ingestion")

In [20]:
import re
import requests
from sec_client import get_user_agent

def companies_in_industry(sic):
    """CIKs of companies SEC lists under this industry code (SIC)."""
    r = requests.get(
        "https://www.sec.gov/cgi-bin/browse-edgar",
        params={"action": "getcompany", "SIC": sic, "type": "10-K", "count": "100", "output": "atom"},
        headers={"User-Agent": get_user_agent()},
        timeout=30,
    )
    return re.findall(r"<cik>(\d+)</cik>", r.text)   # pull every <cik>…</cik> out of the text

peers = companies_in_industry("2030")
print(len(peers), peers[:10])


43 ['0001029412', '0001167905', '0000881106', '0000814339', '0001487197', '0000025350', '0000026285', '0001473227', '0001029420', '0001029416']


In [21]:
from flatten_companyfacts import flatten
from sec_client import SECError, fetch_json

# companies label revenue differently, and labels changed over the years
REVENUE_TAGS = [
    "Revenues",
    "RevenueFromContractWithCustomerExcludingAssessedTax",
    "RevenueFromContractWithCustomerIncludingAssessedTax",
    "SalesRevenueNet",
    "SalesRevenueGoodsNet",
]


def annual_revenue(cik, year):
    """Full-year revenue for the fiscal year ending in `year`, as first reported. None if not found."""
    try:
        facts = fetch_json(f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json", get_user_agent())
    except SECError:
        return None                                  # company has no financial data at SEC

    df = flatten(facts)
    rev = df[df["concept"].isin(REVENUE_TAGS) & (df["unit"] == "USD")]
    days = (rev["end"] - rev["start"]).dt.days
    rev = rev[days.between(350, 380) & (rev["end"].dt.year == year)]   # full year, ending in `year`
    if rev.empty:
        return None
    return rev.sort_values("filed")["val"].iloc[0]   # earliest filing = original number


annual_revenue("0001637459", 2018)


np.float64(26268000000.0)

In [22]:
import time

target = annual_revenue("0001637459", 2018)        # the restater's revenue
rows = []
for cik in peers:
    if cik == "0001637459":
        continue                                     # skip Kraft Heinz itself
    rows.append({"cik": cik, "revenue": annual_revenue(cik, 2018)})
    time.sleep(0.2)                                  # stay well under SEC's 10 requests/second

sizes = pd.DataFrame(rows)
sizes["ratio"] = sizes["revenue"] / target           # 1.0 = same size as Kraft Heinz
similar = sizes[sizes["ratio"].between(0.5, 2)]
similar


,cik,revenue,ratio


## find_control: the plan

```
find_control(restater):
    1. same industry   → 4-digit SIC; if nobody fits, widen to 3-digit, then 2-digit
    2. similar size    → revenue 0.5×–2× in the year before its 4.02
    3. clean history   → no 8-K Item 4.02 since 2009
    → return the closest match
```

In [ ]:
# helpers for find_control: industry search (with pages), revenue lookups, filing-history checks
import math
import time

PAUSE = 0.15          # seconds between SEC requests (stay under 10 per second)
_facts_cache = {}     # cik -> flattened companyfacts, so each company is downloaded once
_frames_cache = {}    # year -> {cik: revenue} for every company, from SEC's frames API
_filings_cache = {}   # cik -> table of all its filings (every page)


def companies_in_industry(sic):
    """CIKs of companies SEC lists under this SIC code, following "next page" when a page is full."""
    ciks, start = [], 0
    while True:
        r = requests.get(
            "https://www.sec.gov/cgi-bin/browse-edgar",
            params={"action": "getcompany", "SIC": sic, "type": "10-K",
                    "count": "100", "start": str(start), "output": "atom"},
            headers={"User-Agent": get_user_agent()},
            timeout=30,
        )
        page = re.findall(r"<cik>(\d+)</cik>", r.text)
        ciks += page
        time.sleep(PAUSE)
        if len(page) < 100:      # last page
            return ciks
        start += 100


def companies_in_group(sic, digits):
    """All CIKs whose SIC starts with the same `digits` digits (4 = exact code, 2 = whole family)."""
    if digits == 4:
        return companies_in_industry(sic)
    width = 10 ** (4 - digits)                       # 3 digits -> 10 codes, 2 digits -> 100 codes
    first = int(sic[:digits]) * width
    ciks = []
    for code in range(first, first + width):
        ciks += companies_in_industry(str(code))
    return ciks


def annual_revenue(cik, year):
    """Full-year revenue for the fiscal year ending in `year`, as first reported. None if not found."""
    if cik not in _facts_cache:
        try:
            _facts_cache[cik] = flatten(fetch_json(
                f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json", get_user_agent()))
        except (SECError, KeyError):
            _facts_cache[cik] = None              # no financial data at SEC
        time.sleep(PAUSE)
    df = _facts_cache[cik]
    if df is None:
        return None

    rev = df[df["concept"].isin(REVENUE_TAGS) & (df["unit"] == "USD")]
    days = (rev["end"] - rev["start"]).dt.days
    rev = rev[days.between(350, 380) & (rev["end"].dt.year == year)]
    if rev.empty:
        return None
    return rev.sort_values("filed")["val"].iloc[0]   # earliest filing = original number


def all_revenues(year):
    """Every company's revenue for calendar year `year`, as one dict {cik: revenue}. 5 requests total."""
    if year not in _frames_cache:
        revenues = {}
        for tag in REVENUE_TAGS:
            try:
                frame = fetch_json(
                    f"https://data.sec.gov/api/xbrl/frames/us-gaap/{tag}/USD/CY{year}.json", get_user_agent())
            except SECError:
                continue                          # nobody used this tag that year
            for row in frame["data"]:
                cik = str(row["cik"]).zfill(10)
                revenues[cik] = max(revenues.get(cik, 0), row["val"])
            time.sleep(PAUSE)
        _frames_cache[year] = revenues
    return _frames_cache[year]


def all_filings(cik):
    """Every filing a company ever made (main file + older pages), as one table."""
    if cik not in _filings_cache:
        subs = fetch_json(f"https://data.sec.gov/submissions/CIK{cik}.json", get_user_agent())
        tables = [pd.DataFrame(subs["filings"]["recent"])]
        for page in subs["filings"]["files"]:
            tables.append(pd.DataFrame(fetch_json(f"https://data.sec.gov/submissions/{page['name']}", get_user_agent())))
        filings = pd.concat(tables, ignore_index=True)
        filings.attrs["name"] = subs["name"]
        _filings_cache[cik] = filings
        time.sleep(PAUSE)
    return _filings_cache[cik]


def has_402_since_2009(cik):
    """True if the company filed an 8-K with Item 4.02 in 2009 or later."""
    f = all_filings(cik)
    hits = f[(f["form"] == "8-K") & f["items"].str.contains("4.02", regex=False) & (f["filingDate"] >= "2009-01-01")]
    return not hits.empty


def still_filing_after(cik, date):
    """True if the company's last filing is on or after `date` (it didn't disappear before then)."""
    return all_filings(cik)["filingDate"].max() >= date


def company_name(cik):
    return all_filings(cik).attrs["name"]

In [ ]:
# the pseudocode above, as real code
def find_control(restater_cik, sic, item_402_date, exclude=()):
    """Closest-size company in the same industry, no Item 4.02 since 2009, still filing a year
    after the restater's 4.02. None if nobody fits."""
    # size year: the year before the 4.02; step back if revenue is missing, zero or negative
    year = int(item_402_date[:4]) - 1
    target = annual_revenue(restater_cik, year)
    if target is None or target <= 0:
        year -= 1
        target = annual_revenue(restater_cik, year)
    if target is None or target <= 0:
        print(f"no usable revenue for {restater_cik}")
        return None
    revenues = all_revenues(year)                     # everyone else: one lookup table

    # the control must still exist for the 12 months we check after the 4.02
    must_file_until = str(pd.Timestamp(item_402_date) + pd.DateOffset(months=12))[:10]

    for digits in (4, 3, 2):                          # widen the industry only if needed
        candidates = set(companies_in_group(sic, digits)) - {restater_cik} - set(exclude)

        sized = [(cik, revenues[cik] / target) for cik in candidates
                 if cik in revenues and 0.5 <= revenues[cik] / target <= 2]

        # closest size first; log makes "half as big" and "twice as big" equally far from 1
        sized.sort(key=lambda pair: abs(math.log(pair[1])))

        for cik, ratio in sized:
            if not has_402_since_2009(cik) and still_filing_after(cik, must_file_until):
                return {"cik": cik, "name": company_name(cik), "ratio": round(ratio, 2),
                        "sic_digits": digits, "year": year}
    return None


find_control("0001637459", "2030", "2019-05-06")   # check: Kraft Heinz -> Mondelez

In [25]:
# run find_control for every restater in the CSV; each control can only be used once
test_companies = pd.read_csv("../dbt/seeds/test_companies.csv", dtype={"cik": str, "sic": str})
restaters = test_companies[test_companies["role"] == "restater"]

controls, used = [], set()
for _, r in restaters.iterrows():
    match = find_control(r["cik"], r["sic"], r["item_402_date"], exclude=used)
    print(f"{r['name']:35} -> {match['name'] if match else 'NO MATCH'}")
    if match:
        used.add(match["cik"])
        controls.append({**match, "paired_with_cik": r["cik"], "paired_with": r["name"]})

pd.DataFrame(controls)

Kraft Heinz Co                      -> Mondelez International, Inc.
Plug Power Inc                      -> NO MATCH
Guidewire Software                  -> Pivotal Software, Inc.
WillScot Holdings                   -> H&E Equipment Services, Inc.
CytomX Therapeutics                 -> Spero Therapeutics, Inc.
Sypris Solutions                    -> MESA LABORATORIES INC /CO/
Acadia Realty Trust                 -> HERSHA HOSPITALITY TRUST
Pursuit Attractions & Hospitality   -> NEW ROYAL HOLDCO I INC.
PEDEVCO                             -> BARNWELL INDUSTRIES INC
Aeon Global Health                  -> Fulgent Genetics, Inc.


,cik,name,ratio,sic_digits,year,paired_with_cik,paired_with
0,0001103982,"Mondelez International, Inc.",0.99,2,2018,0001637459,Kraft Heinz Co
1,0001574135,"Pivotal Software, Inc.",0.99,4,2018,0001528396,Guidewire Software
2,0001339605,"H&E Equipment Services, Inc.",0.74,4,2020,0001647088,WillScot Holdings
3,0001701108,"Spero Therapeutics, Inc.",1.01,4,2022,0001501989,CytomX Therapeutics
4,0000724004,MESA LABORATORIES INC /CO/,1.17,4,2018,0000864240,Sypris Solutions
5,0001063344,HERSHA HOSPITALITY TRUST,1.01,4,2021,0000899629,Acadia Realty Trust
6,0001071255,NEW ROYAL HOLDCO I INC.,1.00,4,2022,0000884219,Pursuit Attractions & Hospitality
7,0000010048,BARNWELL INDUSTRIES INC,0.95,4,2022,0001141197,PEDEVCO
8,0001674930,"Fulgent Genetics, Inc.",1.31,4,2018,0000885074,Aeon Global Health
